# Inverse B: bunch → the four settings

An average of 5 MLPs gives the answer (what R² scores); a re-centred mixture around it gives the probabilities (`models/inverse_b_regress_mdn.py`). Trained on 80% of the configs, scored on 20%, then retrained on all and used for the submission, clipped to the settings' range.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import copy
import shutil

import numpy as np
import pandas as pd

import pallas_pipeline as pp
import pallas_score

project = pp.connect(pp.PROJECT)
fs = project.get_feature_store()
case = pp.CASES["inverse_B"]
module = pp.load_model_file("models/inverse_b_regress_mdn.py")
name, seed, register = "m_inverse_b_inverse_b_regress_mdn", 12, False


def mixture_scores(est, X, rows, n_draw=2000):
    """NLL (standardised units), 90% interval coverage and width of the mixture."""
    w, mu, sd = est.predict_mixture(np.asarray(X, float))
    true = rows[case.targets].to_numpy(float)
    log_n = (-0.5 * ((true[:, None] - mu) / sd) ** 2 - np.log(sd / est.y_scaler.scale_) - 0.5 * np.log(2 * np.pi)).sum(2)
    top = log_n.max(1, keepdims=True)
    rng = np.random.default_rng(0)
    pick = np.minimum((rng.random((len(w), n_draw, 1)) > w.cumsum(1)[:, None, :]).sum(2), w.shape[1] - 1)
    r = np.arange(len(w))[:, None]
    draws = mu[r, pick] + sd[r, pick] * rng.standard_normal((len(w), n_draw, len(case.targets)))
    low, high = np.quantile(draws, 0.05, 1), np.quantile(draws, 0.95, 1)
    return {"nll": -(top[:, 0] + np.log((w * np.exp(log_n - top)).sum(1))).mean(),
            **{f"cover90_{pp.fs_name(t)}": c for t, c in zip(case.targets, ((true >= low) & (true <= high)).mean(0))},
            **{f"width90_{pp.fs_name(t)}": v for t, v in zip(case.targets, (high - low).mean(0))}}

/home/mlenivenko/miniconda3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-10-15 16:18:07,155 INFO: Initializing external client


2026-10-15 16:18:07,156 INFO: Base URL: https://aissai26.hops.live:443


2026-10-15 16:18:08,441 INFO: Python Engine initialized.



Logged in to project, explore it here https://aissai26.hops.live:443/p/2170


## Rows and split

In [2]:
fv = pp.get_feature_view(fs, "m_inverse_b_raw_fv", 1) or pp.create_feature_view(
    fs, case, "m_inverse_b_raw_fv", 1, {c: "none" for c in case.inputs}, description="inverse_B, raw inputs")
td_version = pp.prepare_statistics(fv)
rows = pp.read_rows(fv, case)
train_configs, holdout_configs, _ = pp.split_configs(rows, case, "random", 0.2, seed)
train = rows[rows.config.isin(train_configs)].reset_index(drop=True)
holdout = rows[rows.config.isin(holdout_configs)].reset_index(drop=True)
X_train, X_holdout = pp.transform(fv, train, case), pp.transform(fv, holdout, case)
print(f"{len(train)} train configs (a quarter of them picks the epochs), {len(holdout)} holdout configs")

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (0.64s) 
2306 train configs (a quarter of them picks the epochs), 576 holdout configs


## Train and score the holdout

In [3]:
params = {**module.PARAMS, "columns": list(X_train.columns)}
model = pp.SurrogateModel(module.build_model(**params), case.targets, "scorer").fit(X_train, train)
scores = pallas_score.score_submission(pp.predictions(case, model, X_holdout, holdout), pp.answer_key(case, holdout))
metrics = {"holdout_mean_r2": scores["mean"], **{f"holdout_r2_{pp.fs_name(t)}": scores[t] for t in case.targets},
           **{f"holdout_{k}": v for k, v in mixture_scores(model.estimator, X_holdout, holdout).items()}}
pd.Series(metrics).round(4)

holdout_mean_r2               0.9208
holdout_r2_p_max              0.9775
holdout_r2_cn2_max            0.9629
holdout_r2_l_inj              0.9491
holdout_r2_dip_frac           0.7939
holdout_nll                  -1.7870
holdout_cover90_p_max         0.8906
holdout_cover90_cn2_max       0.9236
holdout_cover90_l_inj         0.9132
holdout_cover90_dip_frac      0.9531
holdout_width90_p_max       412.7922
holdout_width90_cn2_max       0.0360
holdout_width90_l_inj         0.0809
holdout_width90_dip_frac      0.2000
dtype: float64

## Retrain on all configurations, register

Same epochs, no validation; the metrics are the holdout ones above.

In [4]:
model_all = copy.deepcopy(model)
X_all = pp.transform(fv, rows, case)[list(X_train.columns)]
model_all.estimator.refit(np.asarray(X_all, float), np.column_stack([model_all._to_model(t, rows[t]) for t in case.targets]))
versions = {}
for this, label in [(model, name), (model_all, name + "_all")]:
    if register:
        info = {"case": "inverse_B", "inputs": case.inputs, "targets": case.targets, "feature_columns": list(X_train.columns),
                "feature_view": fv.name, "feature_view_version": fv.version, "td_version": td_version,
                "model_file": "inverse_b_regress_mdn", "params": params, "target_scaling": "scorer", "seed": seed,
                "holdout_configs": holdout_configs}
        bundle = pp.save_bundle(this, "models/inverse_b_regress_mdn.py", info, Path("team_models") / label)
        versions[label] = pp.register_model(project, bundle, label, fv, td_version, metrics,
                                            f"inverse_B: 5 MLPs + re-centred mixture k={params['k']}, random holdout 0.2 (seed {seed})").version
        shutil.rmtree(bundle)

## Submission, loaded back from the registry

The answer clipped to the settings' range, and the mixture (weight, centre and width of each solution) next to it.

In [5]:
version = versions.get(name + "_all", 1)
loaded, info = pp.load_registered_model(project, name + "_all", version, "team_models")
test = pd.read_parquet(pp.DATA_DIR / case.test_file)
X_test = pp.transform(fv, test, case)[info["feature_columns"]]
submission = pp.predictions(case, loaded, X_test, test)
submission[case.targets] = submission[case.targets].clip(rows[case.targets].min(), rows[case.targets].max(), axis=1)
folder = Path("submissions/inverse_B") / f"{name}_all_v{version}"
print("wrote", pp.write_submission(case, submission, test, folder / case.submission_file))
w, mu, sd = loaded.estimator.predict_mixture(np.asarray(X_test, float))
parts = [pd.DataFrame({"config": test.config.to_numpy(), "solution": j, "weight": w[:, j],
                       **{f"{t}_{kind}": a[:, j, i] for i, t in enumerate(case.targets) for kind, a in [("centre", mu), ("width", sd)]}})
         for j in range(w.shape[1])]
pd.concat(parts).sort_values(["config", "solution"]).to_csv(folder / "mixture_inverse_B.csv", index=False)

Downloading: 0.000%|          | 0/122 elapsed<00:00 remaining<?

Downloading: 100.000%|██████████| 122/122 elapsed<00:00 remaining<00:00

Downloading: 0.000%|          | 0/27702 elapsed<00:00 remaining<?

Downloading: 100.000%|██████████| 27702/27702 elapsed<00:00 remaining<00:00

Downloading: 0.000%|          | 0/20406 elapsed<00:00 remaining<?

...

Downloading: 88.337%|████████▊ | 3718862/4209867 elapsed<00:00 remaining<00:00

Downloading: 100.000%|██████████| 4209867/4209867 elapsed<00:00 remaining<00:00

Downloading: 0.000%|          | 0/9408 elapsed<00:00 remaining<?

Downloading: 100.000%|██████████| 9408/9408 elapsed<00:00 remaining<00:00

wrote submissions/inverse_B/m_inverse_b_inverse_b_regress_mdn_all_v1/submission_inverse_B.csv
